# How does environmental stochasticity affect Q-learning convergence?

We study movement reliability

$$p\in\{1.0,0.95,0.90,0.80,0.70,0.60\}$$

with exact $Q^*$ available separately for every stationary environment. The
primary target is not merely return: we retain $\|Q_t-Q^*\|_\infty$, L2
error, tie-aware policy disagreement, visitation, success, and online TD
summaries. All uncertainty summaries treat the trial—not an episode—as the
independent experimental unit, with the seed retained as a pairing label.

This notebook uses Protocol v2 throughout. Exploratory training episodes and
update-free held-out evaluations are different tables. Scientific
conditions have stable `condition_id` values; seeded replicates have stable
`trial_id` values. Raw transitions are deterministically sampled while
episode and state-action summaries still observe every transition.

`QUICK=True` is a structural run. Set it to `False` for the specified
100-seed, 5,000-episode experiment, or run the versioned YAML configuration
from the command line.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rllab.agents import LinearDecaySchedule, QLearningAgent
from rllab.environments import Hazard, MazeAction, StochasticMazeEnv
from rllab.evaluation import compare_to_optimal, episodes_to_threshold
from rllab.experiments import Experiment, ExperimentConfig, RunStore, estimate_run
from rllab.metrics import td_error_summary
from rllab.theory import value_iteration
from rllab.visualization import (
    plot_final_distribution,
    plot_learning_curves,
    plot_maze,
    plot_policy,
    plot_state_heatmap,
    plot_sweep_response,
    plot_td_error_heatmap,
    plot_transition_noise,
)

SMOKE = os.environ.get("RL_LAB_NOTEBOOK_SMOKE") == "1"
QUICK = True
SEED = 29
RELIABILITIES = (1.0, 0.70) if SMOKE else (1.0, 0.95, 0.90, 0.80, 0.70, 0.60)
N_SEEDS = 1 if SMOKE else (3 if QUICK else 100)
EPISODES = 8 if SMOKE else (180 if QUICK else 5_000)
EVALUATION_INTERVAL = 4 if SMOKE else (30 if QUICK else 250)
EVALUATION_EPISODES = 1 if SMOKE else (3 if QUICK else 10)
STEP_SAMPLE_FRACTION = 0.25 if SMOKE else 0.05
RESULTS_DIR = Path(os.environ.get("RL_LAB_NOTEBOOK_RESULTS", "results"))
reliability_column = "sweep_environment_action_reliability"
available_styles = set(plt.style.available)
plot_style = next(
    (
        style
        for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot")
        if style in available_styles
    ),
    "default",
)
plt.style.use(plot_style)


## 1. Registered experimental design

The sweep expands a Cartesian product of environment parameters, algorithms,
and seeds. `scenario_id` identifies environment semantics, `condition_id`
identifies a seed-independent scientific condition, and `trial_id` identifies
one seeded replicate. Changing worker count, output location, or diagnostic
retention does not silently create a new scientific condition.

A root seed spawns independent environment, agent, and held-out evaluation
streams. The evaluation policy is cloned and receives no updates. We evaluate
both the matched training environment and a deterministic probe, always on
paired seeds at each checkpoint.

The step-size remains constant here. That is intentional: in a stochastic MDP
it yields a visible asymptotic noise floor and preserves capacity to track
later nonstationarity. A decaying step size addresses a different estimand.


In [ ]:
base_environment = {
    "shape": (6, 8),
    "start": (5, 0),
    "goals": {(0, 7): 8.0},
    "blocked_cells": [(1, 1), (1, 2), (2, 5), (3, 2), (4, 5)],
    "static_walls": [
        ((4, 1), (4, 2)),
        ((2, 3), (2, 4)),
        ((1, 6), (2, 6)),
    ],
    "slip_weights": {"left": 0.45, "right": 0.45, "stay": 0.10},
    "step_reward": -0.04,
    "max_episode_steps": 180,
}
config = ExperimentConfig.from_mapping(
    {
        "config_schema_version": 2,
        "experiment": {
            "name": "q-learning-reliability",
            "episodes": EPISODES,
            "seeds": list(range(N_SEEDS)),
            "environment": {
                "name": "stationary-maze",
                "kind": "stochastic_maze",
                "parameters": base_environment,
            },
            "agents": [
                {
                    "name": "q_learning",
                    "kind": "q_learning",
                    "parameters": {
                        "learning_rate": 0.14,
                        "gamma": 0.98,
                        "epsilon": {
                            "kind": "linear",
                            "start": 0.30,
                            "end": 0.03,
                            "duration": max(1_000, EPISODES * 20),
                        },
                    },
                }
            ],
            "sweep": {"environment.action_reliability": list(RELIABILITIES)},
            "snapshot_interval": 2 if SMOKE else (10 if QUICK else 25),
            "exact_reference": True,
            "tags": {"question": "stochasticity-and-q-convergence"},
        },
        "policy_evaluation": {
            "enabled": True,
            "interval_episodes": EVALUATION_INTERVAL,
            "episodes_per_checkpoint": EVALUATION_EPISODES,
            "include_initial": True,
            "include_final": True,
            "scenarios": [
                {"name": "matched_training_environment"},
                {
                    "name": "deterministic_probe",
                    "environment_overrides": {"action_reliability": 1.0},
                },
            ],
        },
        "execution": {
            "parallel_workers": 1 if QUICK else 8,
            "failure_policy": "fail_fast",
        },
        "artifacts": {
            "output_dir": str(RESULTS_DIR),
            "table_format": "auto",
            "flush_rows": 250 if SMOKE else 10_000,
            "save_q_snapshots": True,
            "step_retention": {
                "mode": "sample",
                "fraction": STEP_SAMPLE_FRACTION,
                "keep_terminal": True,
                "keep_events": True,
            },
        },
    }
)

run_estimate = estimate_run(config)
design = pd.DataFrame(
    {
        "scenario_id": trial.scenario_id,
        "condition_id": trial.condition_id,
        "trial_id": trial.trial_id,
        "seed": trial.seed,
        reliability_column: trial.sweep_values["environment.action_reliability"],
    }
    for trial in config.trials()
)
assert design["trial_id"].is_unique
assert design.groupby("condition_id")[reliability_column].nunique().eq(1).all()
display(pd.Series(run_estimate.as_dict(), name="preflight estimate"))
display(design.head())

result = Experiment(config).run(persist=True, progress=True)
training_episodes = result.training_episodes
held_out_evaluations = result.evaluations
print(result.experiment_id)
print(result.run_directory)
print(
    "training/evaluation/snapshot rows:",
    len(training_episodes),
    len(held_out_evaluations),
    len(result.snapshots),
)


## 2. The artifact store is part of the protocol

A persisted result is a lazy handle over a versioned run store, not one giant
in-memory frame. Each trial writes bounded table parts into a private attempt
directory; only an atomic commit makes those parts visible. The manifest then
selects exactly one successful attempt per `trial_id` and records checksums,
row counts, source provenance, and the resolved configuration.

Step retention changes storage cost, not the learning process or its online
summaries. Here terminal/event steps are always kept and the remaining steps
are selected by a deterministic hash sample. Temporal analyses such as TD
autocorrelation require a dedicated run with `mode: all`; the sampled main
sweep is appropriate for inspecting individual transitions, not adjacency.


In [ ]:
assert result.run_directory is not None
store = RunStore.open(result.run_directory)
commits = store.committed_attempts()
retention_audit = pd.DataFrame(
    {
        "trial_id": commit.trial_id,
        "observed_steps": commit.metadata["observed_steps"],
        "retained_steps": commit.metadata["retained_steps"],
        "retained_fraction": (
            commit.metadata["retained_steps"] / commit.metadata["observed_steps"]
        ),
        "step_parts": sum(
            artifact.table == "steps" for artifact in commit.artifacts
        ),
    }
    for commit in commits
)
assert store.manifest.artifact_schema_version == 2
assert store.manifest.status == "complete"
display(retention_audit.head())

lazy_preview = next(
    result.iter_table(
        "training_episodes",
        columns=("condition_id", "trial_id", "seed", "episode", "episode_return"),
        batch_size=5,
        verify=True,
    )
)
display(lazy_preview)


## 3. Training return and held-out return answer different questions

`training_episodes` contains exploratory behavior and update diagnostics.
`evaluations` contains frozen-policy rollouts on seeds disjoint from training;
repeated evaluation episodes are reduced within each trial/checkpoint before
trials are bootstrapped. In quick mode three seeds only exercise the pipeline
and do not support a scientific confidence interval.


In [ ]:
display_columns = [
    "condition_id",
    "trial_id",
    reliability_column,
    "seed",
    "episode",
    "episode_return",
    "success",
    "episode_length",
    "td_error_variance",
]
display(training_episodes[display_columns].head())

evaluation_curves = (
    held_out_evaluations.groupby(
        [
            "condition_id",
            "trial_id",
            "seed",
            reliability_column,
            "evaluation_scenario",
            "checkpoint_episode",
        ],
        as_index=False,
    )
    .agg(episode_return=("episode_return", "mean"), success=("success", "mean"))
)
matched_evaluation_curves = evaluation_curves.loc[
    evaluation_curves["evaluation_scenario"].eq("matched_training_environment")
].copy()
deterministic_probe_summary = (
    evaluation_curves.loc[
        evaluation_curves["evaluation_scenario"].eq("deterministic_probe")
    ]
    .groupby([reliability_column, "checkpoint_episode"], as_index=False)
    .agg(mean_return=("episode_return", "mean"), n_trials=("trial_id", "nunique"))
)
display(deterministic_probe_summary.tail())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot_learning_curves(
    training_episodes,
    metric="episode_return",
    group=reliability_column,
    smooth=min(15, EPISODES),
    individual=not QUICK,
    n_resamples=300 if QUICK else 2_000,
    ax=axes[0],
)
plot_learning_curves(
    matched_evaluation_curves,
    metric="episode_return",
    x="checkpoint_episode",
    group=reliability_column,
    n_resamples=300 if QUICK else 2_000,
    ax=axes[1],
)
axes[0].set_title("Exploratory training return")
axes[1].set_title("Update-free held-out return")
plt.tight_layout()
plt.show()

matched_final = matched_evaluation_curves.copy()
matched_final = matched_final.loc[
    matched_final["checkpoint_episode"]
    .eq(matched_final.groupby("trial_id")["checkpoint_episode"].transform("max"))
].rename(columns={"checkpoint_episode": "episode"})
plot_final_distribution(
    matched_final,
    metric="episode_return",
    group=reliability_column,
    last_episodes=1,
)
plt.title("Trial-level final held-out return")
plt.show()


## 4. Convergence against a different exact $Q^*$ at each reliability

Comparing all learners to the deterministic optimum would confound learning
error with a change in the estimand. The runner freezes each trial's stationary
model, solves it by value iteration, and records norms and policy diagnostics
at Q snapshots.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot_learning_curves(
    result.snapshots.query("episode >= 0"),
    metric="q_error_inf",
    group=reliability_column,
    log_y=True,
    n_resamples=300 if QUICK else 2_000,
    ax=axes[0],
)
plot_learning_curves(
    result.snapshots.query("episode >= 0"),
    metric="policy_disagreement",
    group=reliability_column,
    n_resamples=300 if QUICK else 2_000,
    ax=axes[1],
)
axes[0].set_title(r"$\|Q_t-Q^*\|_\infty$")
axes[1].set_title("Tie-aware policy disagreement")
plt.tight_layout()
plt.show()

threshold = episodes_to_threshold(
    result.snapshots.query("episode >= 0"),
    metric="policy_disagreement",
    threshold=0.10,
    sustain=2,
    groups=("trial_id", reliability_column),
)
display(
    threshold.groupby(reliability_column, as_index=False)
    .agg(
        median_episodes=("episodes_to_threshold", "median"),
        fraction_reached=("reached", "mean"),
    )
    .sort_values(reliability_column, ascending=False)
)


In [ ]:
plot_sweep_response(
    result.snapshots.query("episode >= 0"),
    parameter="environment.action_reliability",
    metric="q_error_inf",
    last_episodes=1,
)
plt.title("Reliability versus final sup-norm error")
plt.show()

plot_sweep_response(
    training_episodes,
    parameter="environment.action_reliability",
    metric="td_error_variance",
    last_episodes=min(100, EPISODES // 3),
)
plt.title("Reliability versus late TD-error variance")
plt.show()


## 5. TD-error distribution and location under bounded retention

For Q-learning,

$$\delta_t=R_{t+1}+\gamma(1-D_{t+1})\max_a Q_t(S_{t+1},a)
  -Q_t(S_t,A_t).$$

Conditional TD variance mixes transition/reward aleatoric noise, changing
value estimates, and under-explored targets. It is therefore a diagnostic,
not automatically an epistemic uncertainty estimator. Online episode and
state-action tables contain moments computed from every update. The raw step
table is smaller: `retention_reason == "sample"` selects a value-independent
hash sample, while terminal/event rows are kept for forensic inspection.

Marginal quantiles can be estimated from the hash-sampled rows. Sign changes,
rolling windows, and autocorrelation cannot: sampling destroys adjacency.
Those require a narrower Protocol-v2 run whose retention mode is `all`.


In [ ]:
lowest_reliability = min(RELIABILITIES)
retained_steps = result.steps
sampled_steps = retained_steps.loc[
    retained_steps[reliability_column].eq(lowest_reliability)
    & retained_steps["retention_reason"].eq("sample")
]
if sampled_steps.empty:
    print("No hash-sampled rows in this tiny run; online summaries remain available.")
else:
    sampled_summary = td_error_summary(sampled_steps, autocorrelation_lags=())
    display(
        sampled_summary[
            [
                "state",
                "action",
                "n_trials",
                "count",
                "mean_td_error",
                "variance_td_error",
                "q05_td_error",
                "q95_td_error",
            ]
        ]
        .sort_values("variance_td_error", ascending=False)
        .head(12)
    )

final_state_actions = (
    result.state_actions.loc[
        result.state_actions[reliability_column].eq(lowest_reliability)
    ]
    .sort_values("episode")
    .groupby(["trial_id", "state", "action"], as_index=False)
    .tail(1)
    .copy()
)
final_state_actions["absolute_td_total"] = (
    final_state_actions["mean_absolute_td_error"] * final_state_actions["td_count"]
)
trial_state_td = (
    final_state_actions.groupby(["trial_id", "state"], as_index=False)
    .agg(absolute_td_total=("absolute_td_total", "sum"), td_count=("td_count", "sum"))
)
trial_state_td["mean_absolute_td_error"] = (
    trial_state_td["absolute_td_total"] / trial_state_td["td_count"]
)
online_td_by_state = (
    trial_state_td.groupby("state", as_index=False)["mean_absolute_td_error"].mean()
)

analysis_env = StochasticMazeEnv(
    **base_environment,
    action_reliability=lowest_reliability,
)
plot_td_error_heatmap(online_td_by_state, analysis_env, statistic="mean_absolute")
plt.title(f"All-update mean absolute TD error at p={lowest_reliability}")
plt.show()


## 6. Spatial heterogeneity: unreliable shortcut, protected detour

We now hold the overall topology fixed and assign poor reliability only to a
short central corridor. The lower route is longer but nearly deterministic
and protected from lateral slips by edge walls. This separates the global
difficulty effect above from a local risk--distance tradeoff.


In [ ]:
corridor_hazards = [
    Hazard((row, column), penalty=-8.0, terminal=True)
    for row in (1, 3)
    for column in range(2, 7)
]
protected_edges = [
    ((3, column), (4, column))
    for column in range(1, 8)
]
corridor_states = {(2, column): 0.60 for column in range(1, 8)}
heterogeneous_parameters = {
    "shape": (5, 9),
    "start": (2, 0),
    "goals": {(2, 8): 8.0},
    "hazards": corridor_hazards,
    "static_walls": protected_edges,
    "action_reliability": 0.97,
    "state_reliability": corridor_states,
    "slip_weights": {"left": 0.5, "right": 0.5},
    "step_reward": -0.06,
    "max_episode_steps": 250,
}
hetero_env = StochasticMazeEnv(**heterogeneous_parameters)
exact_hetero = value_iteration(hetero_env.exact_mdp(), gamma=0.98)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_transition_noise(hetero_env, ax=axes[0], title="Spatial reliability")
plot_policy(
    exact_hetero.policy,
    hetero_env,
    values=exact_hetero.values,
    ax=axes[1],
    title="Exact optimal policy",
)
plt.tight_layout()
plt.show()


In [ ]:
hetero_config = ExperimentConfig.from_mapping(
    {
        "config_schema_version": 2,
        "experiment": {
            "name": "q-learning-heterogeneous",
            "episodes": 6 if SMOKE else (300 if QUICK else 5_000),
            "seeds": list(range(1 if SMOKE else (4 if QUICK else 100))),
            "environment": {
                "name": "risky-shortcut-safe-detour",
                "kind": "stochastic_maze",
                "parameters": heterogeneous_parameters,
            },
            "agents": [
                {
                    "name": "q_learning",
                    "kind": "q_learning",
                    "parameters": {
                        "learning_rate": 0.12,
                        "gamma": 0.98,
                        "epsilon": {
                            "kind": "linear",
                            "start": 0.35,
                            "end": 0.03,
                            "duration": 30_000,
                        },
                    },
                }
            ],
            "snapshot_interval": 2 if SMOKE else (10 if QUICK else 25),
            "exact_reference": True,
        },
        "policy_evaluation": {
            "enabled": True,
            "interval_episodes": 3 if SMOKE else (50 if QUICK else 250),
            "episodes_per_checkpoint": 1 if SMOKE else (3 if QUICK else 10),
            "include_initial": True,
            "include_final": True,
        },
        "execution": {"parallel_workers": 1 if QUICK else 8},
        "artifacts": {
            "output_dir": str(RESULTS_DIR),
            "table_format": "auto",
            "flush_rows": 250 if SMOKE else 10_000,
            "save_q_snapshots": True,
            "step_retention": {
                "mode": "none",
                "keep_terminal": True,
                "keep_events": True,
            },
        },
    }
)
heterogeneous_result = Experiment(hetero_config).run(persist=True)
plot_learning_curves(
    heterogeneous_result.snapshots.query("episode >= 0"),
    metric="policy_disagreement",
    group="agent",
    n_resamples=300 if QUICK else 2_000,
)
plt.title("Learning the local risk--distance tradeoff")
plt.show()

final_heterogeneous_state_actions = (
    heterogeneous_result.state_actions.sort_values("episode")
    .groupby(["trial_id", "state", "action"], as_index=False)
    .tail(1)
)
trial_state_visits = (
    final_heterogeneous_state_actions.groupby(["trial_id", "state"], as_index=False)[
        "visit_count"
    ].sum()
)
state_visits = (
    trial_state_visits.groupby("state", as_index=False)["visit_count"]
    .mean()
    .set_index("state")["visit_count"]
)
plot_state_heatmap(
    state_visits.to_dict(),
    hetero_env,
    colorbar_label="mean visits per seed",
    title="Where Q-learning samples",
)
plt.show()


### Inspect one learned policy without hiding the interaction loop

The registered runner is the reproducible path for a sweep. For close
inspection it is also useful to keep one learner in memory. This loop uses the
same agent/environment contract and demonstrates the exact quantity logged as
a TD residual.


In [ ]:
single_env = StochasticMazeEnv(**heterogeneous_parameters)
single_agent = QLearningAgent(
    single_env.n_states,
    single_env.n_actions,
    learning_rate=0.12,
    gamma=0.98,
    epsilon=LinearDecaySchedule(0.35, 0.03, 20_000),
    seed=SEED,
)
q_errors, policy_errors, td_errors = [], [], []
for episode in range(30 if SMOKE else (500 if QUICK else 5_000)):
    observation, _ = single_env.reset(seed=SEED if episode == 0 else None)
    state = int(observation)
    terminated = truncated = False
    while not (terminated or truncated):
        action = single_agent.act(state)
        observation, reward, terminated, truncated, _ = single_env.step(action)
        next_state = int(observation)
        update = single_agent.update(state, action, reward, next_state, terminated)
        td_errors.append(update.td_error)
        state = next_state
    if episode % 10 == 0:
        diagnostics = compare_to_optimal(single_agent.q_values, exact_hetero.q_values)
        q_errors.append(diagnostics["q_error_inf"])
        policy_errors.append(diagnostics["policy_disagreement"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_policy(
    single_agent.greedy_policy,
    single_env,
    values=single_agent.values,
    ax=axes[0],
    title="Learned greedy policy",
)
axes[1].plot(np.arange(len(q_errors)) * 10, q_errors, label=r"$\|Q-Q^*\|_\infty$")
axes[1].plot(np.arange(len(policy_errors)) * 10, policy_errors, label="policy disagreement")
axes[1].set(xlabel="episode", title="Ground-truth diagnostics")
axes[1].legend()
plt.tight_layout()
plt.show()


## 7. Interpretation and next experiments

Environmental stochasticity changes at least three things simultaneously:
the optimal value/policy, the conditional variance of TD targets, and the
state-action occupancy induced by exploration. Consequently, slower
sup-norm convergence cannot be attributed to "noise" without checking
coverage and local action gaps.

Follow-ups enabled by the stored schema include variance-adaptive
$\alpha_t(s,a)$, forgetting under regime changes, change-point tests on TD
residuals in a targeted full-retention run, model-based uncertainty from
transition counts, and hitting-time rather than discounted-return objectives.
Each should be compared on the same trial-level distributions and, where
meaningful, the same exact model and paired held-out evaluation seeds.
